# 簡易的なコーディングエージェントの実装

コマンド実行やファイルの読み書きができるツールをLLMに与えて、part1_2で実装したエージェントループで動かすと、Claude Codeのようなコーディングエージェントの簡易版になります。

ここでは、次の3つのツールを素のPythonで実装して、Chat Completions APIのFunction callingから呼び出します。

- `run_command`: シェルコマンドを実行する
- `read_file`: ファイルを読む
- `write_file`: ファイルに書く

In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

## 作業ディレクトリ

エージェントが触れる範囲を限定するため、作業ディレクトリを `tmp/coding-agent` に固定します。

> **注意**: コマンド実行やファイル書き込みをLLMに任せると、予期しない操作が行われる可能性があります。講座では仕組みを理解するために使いますが、実際に使う際は動作する環境や権限に十分な注意が必要です。

In [ ]:
from pathlib import Path

WORK_DIR = Path("../tmp/coding-agent").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(WORK_DIR)

## コマンド実行ツールの実装

In [ ]:
import json
import subprocess


def run_command(command: str) -> str:
    """作業ディレクトリでシェルコマンドを実行し、標準出力・標準エラー出力・終了コードを返す"""
    result = subprocess.run(
        command,
        shell=True,
        cwd=WORK_DIR,
        capture_output=True,
        text=True,
        timeout=60,
    )
    return json.dumps(
        {
            "stdout": result.stdout,
            "stderr": result.stderr,
            "returncode": result.returncode,
        },
        ensure_ascii=False,
    )


print(run_command("ls -la"))

## Read・Writeツールの実装

作業ディレクトリの外を読み書きできないように、パスを検査してから処理します。

In [ ]:
def _resolve_path(path: str) -> Path:
    """作業ディレクトリからの相対パスとして解決し、外に出ていたらエラーにする"""
    resolved = (WORK_DIR / path).resolve()
    if not resolved.is_relative_to(WORK_DIR):
        raise ValueError(f"作業ディレクトリの外は操作できません: {path}")
    return resolved


def read_file(path: str) -> str:
    """ファイルの内容を読んで返す"""
    return _resolve_path(path).read_text(encoding="utf-8")


def write_file(path: str, content: str) -> str:
    """ファイルに内容を書き込む（存在すれば上書き）"""
    target = _resolve_path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return f"{path} に {len(content)} 文字を書き込みました"


print(write_file("memo.txt", "hello"))
print(read_file("memo.txt"))

## ツールの定義

Chat Completions APIに渡す `tools`（関数の説明とパラメータのJSON Schema）と、名前から関数を引くための辞書を用意します。

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "run_command",
            "description": "作業ディレクトリでシェルコマンドを実行します。標準出力・標準エラー出力・終了コードをJSONで返します。",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "実行するシェルコマンド"},
                },
                "required": ["command"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "作業ディレクトリ内のファイルの内容を読みます。",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "作業ディレクトリからの相対パス"},
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "作業ディレクトリ内のファイルに内容を書き込みます。存在すれば上書きします。",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "作業ディレクトリからの相対パス"},
                    "content": {"type": "string", "description": "書き込む内容"},
                },
                "required": ["path", "content"],
            },
        },
    },
]

available_functions = {
    "run_command": run_command,
    "read_file": read_file,
    "write_file": write_file,
}

## エージェントループ

part1_2で実装した `agent_loop` と同じものです。LLMがツールを使いたいと応答する限り、ツールを実行して結果を渡し続けます。

In [ ]:
from openai import OpenAI

client = OpenAI()


def agent_loop(
    messages: list,
    tools: list,
    available_functions: dict,
    max_iterations: int = 20,
) -> str | None:
    """LLMがツールを使いたいと応答する限り、ツールを実行して結果を渡し続ける（エージェントループ）"""
    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model="gpt-5.6-luna",
            messages=messages,
            tools=tools,
            reasoning_effort="none",
        )
        response_message = response.choices[0].message
        messages.append(response_message.to_dict())

        # ツールを使わない応答なら、それが最終的な回答
        if not response_message.tool_calls:
            return response_message.content

        # ツールを使いたいという応答なら、ツールを実行して結果をmessagesに追加し、再度LLMを呼び出す
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            print(f"[tool] {function_name}({function_args})")
            function_response = available_functions[function_name](**function_args)
            messages.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                }
            )

    return "（反復回数の上限に達しました）"

## コーディングエージェントを動かす

システムプロンプト（`developer` ロール）で役割と作業ディレクトリの前提を伝えます。

In [ ]:
system_prompt = """あなたはコーディングエージェントです。
作業ディレクトリの中でファイルの作成・編集やコマンドの実行を行い、ユーザーの依頼を達成してください。
作業が終わったら、何をしたかを簡潔に報告してください。"""

messages = [
    {"role": "developer", "content": system_prompt},
    {
        "role": "user",
        "content": "hello.py という名前で、Hello World と表示するPythonプログラムを作成して、実行して結果を教えてください",
    },
]

answer = agent_loop(messages, tools, available_functions)
print(answer)

In [ ]:
messages.append(
    {
        "role": "user",
        "content": "hello.py を読んで、表示するメッセージを日本語の「こんにちは、世界」に変えて保存し、もう一度実行してください",
    }
)

answer = agent_loop(messages, tools, available_functions)
print(answer)

In [ ]:
# 作業ディレクトリに作られたファイルを確認
print(run_command("ls -la && cat hello.py"))

## LangChainのcreate_agentで同じことをする

同じ3つの関数を `@tool` でLangChainのツールにして `create_agent` に渡すと、エージェントループを自分で書かずに同じことができます。

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool

model = init_chat_model(
    model="gpt-5.6-luna",
    model_provider="openai",
    reasoning_effort="none",
)

agent = create_agent(
    model=model,
    tools=[tool(run_command), tool(read_file), tool(write_file)],
    system_prompt=system_prompt,
)

initial_state = {
    "messages": [
        HumanMessage(
            content="fizzbuzz.py という名前で、1から15までのFizzBuzzを表示するプログラムを作成して、実行して結果を教えてください"
        ),
    ]
}

for event in agent.stream(initial_state, stream_mode="updates"):
    for value in event.values():
        latest_message = value["messages"][-1]
        latest_message.pretty_print()

## Deep Agentsへ

ここで作ったのは「ツール3つ + エージェントループ」だけの最小構成です。実際のコーディングエージェントは、これに

- 計画を立てるツール（TODOリスト）
- 独立したタスクを別のコンテキストで実行するサブエージェント
- ファイル操作やコマンド実行に関する詳細なシステムプロンプト
- 人間による承認（Human-in-the-Loop）

などを足したものです。LangChainの [Deep Agents](https://github.com/langchain-ai/deepagents) は、この構成をパッケージにしたものです。動かしてみる場合は `pages/partX_2_deepagents.py` を参照してください。